# 🧪 Crime Data Analysis – Project Scaffold

This notebook is a **production-ready scaffold** tailored to your group questions. It is structured into reusable functions and runnable blocks for:

- Data loading & validation
- Cleaning & feature engineering (time, place, categories)
- Missingness audit
- Temporal trends (monthly, weekday/weekend, hour-of-day)
- Spatial analysis (folium heatmaps, DBSCAN clusters)
- Co-occurrence of offenses (pairs/triads)
- Victim intensity & Incident duration
- Drug possession vs selling by place type
- Vehicle crime risk by area

⚙️ **Assumptions / Column names** (adjust if different):
- `Incident_ID`, `Start_Date_Time`, `End_Date_Time`, `Crime Name1`, `Crime Category`, `Offense_Code`, `Victim_Count`,
  `Police_District`, `Agency`, `City`, `ZIP`, `Place_Type`, `Latitude`, `Longitude`.

📌 Tip: If your dataset uses different names, edit the `COLS` mapping in the next cell.


In [ ]:
# === 0) Imports & Global Config ===
import os, re, math, json, itertools
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns

from dateutil.relativedelta import relativedelta

# Spatial (lightweight, no GeoPandas)
import folium
from folium.plugins import HeatMap, MarkerCluster
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

pd.options.display.max_columns = 120
pd.options.display.width = 160
plt.rcParams['figure.figsize'] = (10,6)

# Column map (edit to match your file)
COLS = {
    'id': 'Incident_ID',
    'start': 'Start_Date_Time',
    'end': 'End_Date_Time',
    'name': 'Crime Name1',
    'cat': 'Crime Category',
    'off': 'Offense Code',
    'victims': 'Victims',
    'dist': 'Police District Name',
    'agency': 'Agency',
    'city': 'City',
    'zip': 'Zip Code',
    'place': 'Place',
    'lat': 'Latitude',
    'lon': 'Longitude',
}

# Time-of-day buckets
TOD_BUCKETS = [
    (0, 6, 'Night'),
    (6, 12, 'Morning'),
    (12, 18, 'Afternoon'),
    (18, 24, 'Evening')
]

def bucket_hour(h):
    for lo, hi, label in TOD_BUCKETS:
        if lo <= h < hi:
            return label
    return 'Unknown'


## 1) Load Data
Set your CSV path(s). Optionally provide a **population CSV** with columns: `Area` (matching `City` or `Police_District`) and `Population` to compute rates per 100k.


In [ ]:
DATA_PATH = 'data/Crime_Dataset_Cleaned.csv'  # <-- TODO: update path
POP_PATH = None               # e.g., 'data/population_by_city.csv' or None

parse_dates = [COLS['start']]
if COLS['end']:
    parse_dates.append(COLS['end'])

cd = pd.read_csv(DATA_PATH, parse_dates=parse_dates, infer_datetime_format=True, low_memory=False)
print(cd.shape)
cd.head(3)

## 2) Basic Cleaning & Feature Engineering
- Coerce datetimes
- Standardize categorical text
- Derive Year/Month/Day/Hour features and weekday/weekend flags
- Create **duration_min** if `End_Date_Time` exists


In [ ]:
def _clean_str(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r'\s+', ' ', s)
    return s.title()

def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Ensure datetime
    df[COLS['start']] = pd.to_datetime(df[COLS['start']], errors='coerce')
    if COLS['end'] in df.columns:
        df[COLS['end']] = pd.to_datetime(df[COLS['end']], errors='coerce')

    # Clean selected strings
    for c in [COLS['name'], COLS['cat'], COLS['dist'], COLS['agency'], COLS['city'], COLS['zip'], COLS['place']]:
        if c in df.columns:
            df[c] = df[c].apply(_clean_str)

    # Coordinates numeric
    for c in [COLS['lat'], COLS['lon']]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    # Temporal derived
    s = df[COLS['start']]
    df['Year'] = s.dt.year
    df['Month'] = s.dt.month
    df['MonthStr'] = s.dt.to_period('M').astype(str)
    df['Day'] = s.dt.day
    df['Hour'] = s.dt.hour
    df['Weekday'] = s.dt.day_name()
    df['IsWeekend'] = s.dt.weekday >= 5
    df['DayType'] = np.where(df['IsWeekend'], 'Weekend', 'Weekday')
    df['TimeOfDay'] = df['Hour'].apply(bucket_hour)

    # Duration
    if COLS['end'] in df.columns:
        dur = (df[COLS['end']] - df[COLS['start']]).dt.total_seconds() / 60.0
        df['duration_min'] = dur
    else:
        df['duration_min'] = np.nan
    return df

cd = prepare_features(cd)
cd.head(3)

## 3) Missingness Audit
Quantify percentage of missing values for key fields and visualize.


In [ ]:
KEYS = [COLS['start'], COLS['end'], COLS['lat'], COLS['lon'], COLS['place'], COLS['victims'], COLS['off'], COLS['cat'], COLS['name']]
avail = {k: (1.0 - cd[k].isna().mean())*100 if k in cd.columns else np.nan for k in KEYS}
miss = pd.Series({k: (100.0 - v) if not np.isnan(v) else np.nan for k, v in avail.items()}).sort_values(ascending=False)
print('Missingness (%):')
display(miss)

ax = miss.plot(kind='bar')
ax.set_title('Missingness by Field (%)')
ax.set_ylabel('% Missing')
plt.tight_layout()
plt.show()

## 4) Temporal Trends (Monthly, Category) – Q2, Q10, Pandemic Effect
- Monthly totals overall & by category
- YOY deltas (2018–2022)
- Pre-2020 vs 2020 vs Post-2020 comparison (Interrupted view)


In [ ]:
def monthly_counts(df, by=None):
    g = df.groupby(['Year','Month'])
    if by:
        g = df.groupby(['Year','Month', by])
    out = g.size().reset_index(name='n')
    out['MonthStr'] = pd.to_datetime(out['Year'].astype(str) + '-' + out['Month'].astype(str) + '-01')
    return out

m_all = monthly_counts(cd)
m_cat = monthly_counts(cd, by=COLS['cat']) if COLS['cat'] in cd.columns else None

# Plot all crimes monthly
fig, ax = plt.subplots()
ax.plot(m_all['MonthStr'], m_all['n'])
ax.set_title('Monthly Crime Counts (All)')
ax.set_xlabel('Month')
ax.set_ylabel('Incidents')
plt.xticks(rotation=45)
plt.tight_layout(); plt.show()

if m_cat is not None:
    for cat, dfc in m_cat.groupby(COLS['cat']):
        fig, ax = plt.subplots()
        ax.plot(dfc['MonthStr'], dfc['n'])
        ax.set_title(f'Monthly Counts – {cat}')
        ax.set_xlabel('Month')
        ax.set_ylabel('Incidents')
        plt.xticks(rotation=45)
        plt.tight_layout(); plt.show()

# Pandemic comparison
def pandemic_buckets(y):
    if y <= 2019:
        return 'Pre-2020'
    elif y == 2020:
        return '2020'
    else:
        return 'Post-2020'

cd['PandemicEra'] = cd['Year'].apply(pandemic_buckets)
era = cd.groupby(['PandemicEra', COLS['cat']]).size().reset_index(name='n') if COLS['cat'] in cd.columns else None
if era is not None:
    pivot = era.pivot(index='PandemicEra', columns=COLS['cat'], values='n').fillna(0)
    display(pivot)


## 5) Weekday/Weekend & Hour-of-Day Patterns – Q4, Q6, Q9
Heatmap of counts by hour × weekday; split by **Top 2 districts** (weekday vs weekend).


In [ ]:
def plot_hour_weekday_heatmap(df, title='Hour × Weekday Crime Heatmap'):
    p = df.pivot_table(index='Weekday', columns='Hour', values=COLS['id'] if COLS['id'] in df.columns else COLS['start'], aggfunc='count').reindex(
        ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    )
    sns.heatmap(p, cmap='viridis')
    plt.title(title)
    plt.ylabel('Weekday'); plt.xlabel('Hour')
    plt.show()

plot_hour_weekday_heatmap(cd)

top_d = cd[COLS['dist']].value_counts().head(2).index.tolist() if COLS['dist'] in cd.columns else []
for d in top_d:
    sub = cd[cd[COLS['dist']] == d]
    plot_hour_weekday_heatmap(sub[sub['DayType']=='Weekday'], title=f'{d}: Weekday Heatmap')
    plot_hour_weekday_heatmap(sub[sub['DayType']=='Weekend'], title=f'{d}: Weekend Heatmap')


## 6) District/City Mix – Q1, Q3, Q10
Top crimes per district/city and ranked volumes.


In [ ]:
def top_types_by_area(df, area_col, type_col, topn=5):
    g = df.groupby([area_col, type_col]).size().reset_index(name='n')
    ranks = g.sort_values(['%s' % area_col, 'n'], ascending=[True, False])
    top = ranks.groupby(area_col).head(topn)
    return top

if COLS['dist'] in cd.columns:
    top_by_dist = top_types_by_area(cd, COLS['dist'], COLS['name'])
    display(top_by_dist.head(20))

if COLS['city'] in cd.columns:
    top_by_city = top_types_by_area(cd, COLS['city'], COLS['name'])
    display(top_by_city.head(20))


## 7) Victim Intensity – Q5
Which categories have the highest average victims?


In [ ]:
if COLS['victims'] in cd.columns and COLS['cat'] in cd.columns:
    vc = cd.groupby(COLS['cat'])[COLS['victims']].agg(['count','mean','median']).sort_values('mean', ascending=False)
    display(vc)
    vc['mean'].plot(kind='bar'); plt.title('Avg Victims by Category'); plt.ylabel('Mean Victims'); plt.show()
else:
    print('Victim count or category column missing – skipping.')


## 8) Incident Duration – Q8 (if End time exists)
Distribution of `duration_min` overall and by crime type; missingness summary for end times.


In [ ]:
if COLS['end'] in cd.columns:
    miss_end = cd[COLS['end']].isna().mean()*100
    print(f"% missing End_Date_Time: {miss_end:.2f}%")
    cd['duration_min'].plot(kind='hist', bins=50)
    plt.title('Incident Duration (minutes)'); plt.xlabel('Minutes'); plt.show()
    if COLS['name'] in cd.columns:
        bx = cd[[COLS['name'], 'duration_min']].dropna()
        top_types = bx[COLS['name']].value_counts().head(10).index
        sns.boxplot(data=bx[bx[COLS['name']].isin(top_types)], x=COLS['name'], y='duration_min')
        plt.xticks(rotation=45); plt.title('Duration by Top Crime Types'); plt.show()
else:
    print('No End time column – duration skipped.')


## 9) Offense Co-occurrence – Q2/Q3 (combinations)
Frequent pairs and triads per incident; optional split by place type.


In [ ]:
def offense_itemsets(df, by_place=False, min_support=0.005):
    if COLS['id'] not in df.columns or COLS['off'] not in df.columns:
        print('Missing Incident_ID or Offense_Code')
        return None
    res = []
    if by_place and COLS['place'] in df.columns:
        groups = df.groupby(COLS['place'])
    else:
        groups = [(None, df)]
    for place, g in groups:
        basket = g.groupby(COLS['id'])[COLS['off']].apply(lambda s: sorted(set([str(x) for x in s.dropna().tolist()]))).tolist()
        N = len(basket)
        # Count pairs and triads
        pair_counter = Counter()
        tri_counter = Counter()
        for items in basket:
            for a,b in itertools.combinations(items, 2):
                pair_counter[(a,b)] += 1
            for a,b,c in itertools.combinations(items, 3):
                tri_counter[(a,b,c)] += 1
        def to_df(counter, k):
            rows = [(list(t), v, v/max(1,N)) for t,v in counter.items()]
            out = pd.DataFrame(rows, columns=['items','count','support'])
            out = out[out['support'] >= min_support].sort_values('support', ascending=False)
            out['k'] = k
            out['Place'] = place
            return out
        df_pairs = to_df(pair_counter, 2)
        df_tris = to_df(tri_counter, 3)
        res.append(pd.concat([df_pairs, df_tris], ignore_index=True))
    return pd.concat(res, ignore_index=True)

co_all = offense_itemsets(cd, by_place=False, min_support=0.002)
display(co_all.head(20) if co_all is not None else 'No co-occurrence output')

co_by_place = offense_itemsets(cd, by_place=True, min_support=0.005)
display(co_by_place.head(20) if co_by_place is not None else 'No co-occurrence by place output')


## 10) Drug Possession vs Selling by Place Type – Q8 (domain-specific)
Keyword split using `Crime Name1` and/or `Offense_Code`. Adjust keyword lists for your taxonomy.


In [ ]:
POS_KEYS = ['possession', 'poss', 'use']
SELL_KEYS = ['sell', 'sales', 'distribution', 'distribute', 'manufacture', 'traffick']

def label_drug_type(name, code):
    text = f"{name} {code}".lower()
    if any(k in text for k in SELL_KEYS):
        return 'Selling/Distribution'
    if any(k in text for k in POS_KEYS):
        return 'Possession/Use'
    return 'Unclear/Other'

if COLS['name'] in cd.columns and COLS['off'] in cd.columns:
    cd['DrugSubtype'] = cd.apply(lambda r: label_drug_type(r.get(COLS['name'], ''), r.get(COLS['off'], '')), axis=1)
    if COLS['place'] in cd.columns:
        t = cd.groupby([COLS['place'], 'DrugSubtype']).size().reset_index(name='n')
        t['share'] = t.groupby(COLS['place'])['n'].apply(lambda s: s/s.sum())
        display(t.sort_values(['share'], ascending=False).head(30))
        # 100% stacked bar plot (top places)
        top_places = cd[COLS['place']].value_counts().head(10).index
        pvt = t[t[COLS['place']].isin(top_places)].pivot(index=COLS['place'], columns='DrugSubtype', values='share').fillna(0)
        pvt.plot(kind='bar', stacked=True)
        plt.title('Drug Subtypes by Place (share)'); plt.ylabel('Share'); plt.legend(bbox_to_anchor=(1.05,1), loc='upper left')
        plt.tight_layout(); plt.show()
else:
    print('Missing columns for drug analysis.')


## 11) Vehicle Crime Risk by Area – Q12/Q5
Filter to vehicle-related crimes; rank by counts and (if available) per-100k population rates.


In [ ]:
VEHICLE_KEYWORDS = ['motor vehicle theft','vehicle theft','auto theft','theft from vehicle','theft from auto','burglary from vehicle']

def is_vehicle_crime(name):
    txt = str(name).lower()
    return any(k in txt for k in VEHICLE_KEYWORDS)

area_col = COLS['city'] if COLS['city'] in cd.columns else COLS['dist']
veh = cd[cd[COLS['name']].apply(is_vehicle_crime)] if COLS['name'] in cd.columns else pd.DataFrame()

if not veh.empty:
    rank = veh.groupby(area_col).size().reset_index(name='n').sort_values('n', ascending=False)
    display(rank.head(20))
    if POP_PATH:
        pop = pd.read_csv(POP_PATH)
        # Expect columns: Area, Population
        pop.columns = [c.strip().title() for c in pop.columns]
        pop = pop.rename(columns={'Area':'AreaKey', 'Population':'Population'})
        rank = rank.rename(columns={area_col:'AreaKey'})
        joined = rank.merge(pop, on='AreaKey', how='left')
        joined['rate_per_100k'] = (joined['n'] / joined['Population']) * 1e5
        display(joined.sort_values('rate_per_100k', ascending=False).head(20))
else:
    print('No vehicle-crime rows found – adjust keywords or column mapping.')


## 12) Spatial Maps – Heatmaps & Clusters – Q7, Q12
Interactive maps with **folium**. Generates:
- Overall heatmap (optionally by category)
- DBSCAN clusters for a selected filter (e.g., `Crime Category == Crime Against Property`)


In [ ]:
def map_center(df):
    lat, lon = df[COLS['lat']].median(), df[COLS['lon']].median()
    if np.isnan(lat) or np.isnan(lon):
        lat, lon = 38.9907, -77.0261  # default center (e.g., Montgomery County, MD)
    return lat, lon

def folium_heatmap(df, title='Crime Heatmap', out_html='heatmap.html', filt=None):
    d = df.copy()
    if filt is not None:
        d = d.query(filt)
    d = d[[COLS['lat'], COLS['lon']]].dropna().copy()
    if d.empty:
        print('No coordinates to map')
        return None
    lat, lon = map_center(df)
    m = folium.Map(location=[lat, lon], zoom_start=10, tiles='cartodbpositron')
    HeatMap(d.values.tolist(), radius=10, blur=12, max_zoom=13).add_to(m)
    m.save(out_html)
    return m

def folium_dbscan_clusters(df, eps_meters=250, min_samples=20, out_html='dbscan.html', filt=None):
    # Filter and standardize
    d = df.copy()
    if filt is not None:
        d = d.query(filt)
    d = d[[COLS['lat'], COLS['lon']]].dropna().copy()
    if d.empty:
        print('No coordinates for clustering')
        return None
    # Approximate meter scaling for lat/lon near mid-latitudes
    # 1 degree lat ~ 111_320 m; 1 degree lon ~ 111_320 * cos(lat)
    lat0 = d[COLS['lat']].mean()
    scale_lat = 111_320.0
    scale_lon = 111_320.0 * math.cos(math.radians(lat0))
    X = d[[COLS['lat'], COLS['lon']]].values.copy()
    X_scaled = np.column_stack([(X[:,0] * scale_lat), (X[:,1] * scale_lon)])
    db = DBSCAN(eps=eps_meters, min_samples=min_samples).fit(X_scaled)
    d['cluster'] = db.labels_
    n_clust = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    print(f'Clusters found: {n_clust}')

    lat, lon = map_center(d)
    m = folium.Map(location=[lat, lon], zoom_start=11, tiles='cartodbpositron')
    cl = MarkerCluster().add_to(m)
    for (la, lo, c) in d[[COLS['lat'], COLS['lon'], 'cluster']].itertuples(index=False):
        col = 'red' if c == -1 else 'blue'
        folium.CircleMarker([la, lo], radius=2, color=col, fill=True, fill_opacity=0.6).add_to(cl)
    m.save(out_html)
    return m

# Example usage (uncomment and adjust filter):
# folium_heatmap(cd, out_html='all_heatmap.html')
# folium_heatmap(cd, out_html='society_heatmap.html', filt=f"`{COLS['cat']}` == 'Crime Against Society'")
# folium_dbscan_clusters(cd, eps_meters=300, min_samples=25, out_html='clusters_property.html', filt=f"`{COLS['cat']}` == 'Crime Against Property'")


## 13) City/District Choropleth (Optional)
If you have a shapefile/GeoJSON for districts or cities, you can load that into folium and choropleth by counts/rates. This scaffold stays lightweight and omits GeoPandas for easier Mac install.


## 14) QA: Assumptions & Limitations Notes
- Ensure incident-level vs offense-level counting consistency (group by `Incident_ID` where needed).
- For pandemic effect, prefer **rates** if population changed; otherwise report counts with caveats.
- For place-type comparisons, standardize values (merge synonyms).
- For co-occurrence, beware rare codes—control with `min_support` and FDR if multiple testing is added.
